In [1]:
import os
from dotenv import load_dotenv
load_dotenv()

gemini_api_key = os.getenv("GEMINI_API_KEY")

In [2]:
from langchain_google_genai import ChatGoogleGenerativeAI

llm = ChatGoogleGenerativeAI(model="gemini-2.5-flash", google_api_key=gemini_api_key)

d:\jsh\langchain-workspace\ch04\.venv\Lib\site-packages\langchain_core\_api\deprecation.py:26: UserWarning: Core Pydantic V1 functionality isn't compatible with Python 3.14 or greater.
  from pydantic.v1.fields import FieldInfo as FieldInfoV1


In [3]:
llm.invoke("대한민국의 수도는?")

AIMessage(content='대한민국의 수도는 **서울**입니다.', additional_kwargs={}, response_metadata={'prompt_feedback': {'block_reason': 0, 'safety_ratings': []}, 'finish_reason': 'STOP', 'model_name': 'gemini-2.5-flash', 'safety_ratings': [], 'grounding_metadata': {}, 'model_provider': 'google_genai'}, id='lc_run--ed035791-0e02-41ac-bf48-bf8ecfb68155-0', usage_metadata={'input_tokens': 7, 'output_tokens': 175, 'total_tokens': 182, 'input_token_details': {'cache_read': 0}, 'output_token_details': {'reasoning': 165}})

### 1. Message passing

In [4]:
from langchain_core.prompts import ChatPromptTemplate

prompt = ChatPromptTemplate.from_messages(
    [
        (
            "system",
            "You are a helpful assistant. Answer all questions to the best of your ability. You must answer in Korean.",
        ),
        ("placeholder", "{messages}"),
    ]
)

chain = prompt | llm

#### 튜플

In [5]:
ai_msg = chain.invoke(
    {
        "messages": [
            (
                "human",
                "내 이름은 김일남이야. 나이는 99세야.",
            ),
            ("ai",  "그렇군요. 나이가 많으시네요!"),
            ("human", "내 나이는?"),
        ],
    }
)

In [6]:
ai_msg

AIMessage(content='이전에 말씀해주셨던 나이는 **99세**입니다.', additional_kwargs={}, response_metadata={'prompt_feedback': {'block_reason': 0, 'safety_ratings': []}, 'finish_reason': 'STOP', 'model_name': 'gemini-2.5-flash', 'safety_ratings': [], 'grounding_metadata': {}, 'model_provider': 'google_genai'}, id='lc_run--f787ea85-c693-4947-ae04-5203169835e5-0', usage_metadata={'input_tokens': 56, 'output_tokens': 81, 'total_tokens': 137, 'input_token_details': {'cache_read': 0}, 'output_token_details': {'reasoning': 66}})

In [7]:
print(ai_msg.content)

이전에 말씀해주셨던 나이는 **99세**입니다.


In [10]:
history_list = []

while(True):
    user_input = input()

    if user_input == "종료": break
    
    history_list.append(
        (
            "human",
            user_input,
        )
    )
    
    print("CHAT_HISTORY:", history_list)
    
    ai_msg = chain.invoke(
        {
            "messages": history_list,
        }
    )
    
    print("AI:", ai_msg.content)

    history_list.append(
        (
            "ai",
            ai_msg.content,
        )
    )

CHAT_HISTORY: [('human', 'ㄴㅇㅁ')]
AI: 죄송합니다만, 'ㄴㅇㅁ'은(는) 제가 아는 한국어 단어나 표현이 아닙니다. 혹시 다른 것을 말씀하시려던 건가요? 좀 더 자세히 알려주시면 도와드릴 수 있습니다.
CHAT_HISTORY: [('human', 'ㄴㅇㅁ'), ('ai', "죄송합니다만, 'ㄴㅇㅁ'은(는) 제가 아는 한국어 단어나 표현이 아닙니다. 혹시 다른 것을 말씀하시려던 건가요? 좀 더 자세히 알려주시면 도와드릴 수 있습니다."), ('human', 'ㅎㅇ')]
AI: 안녕하세요! (Hello!)


#### 객체

In [11]:
from langchain_core.messages import HumanMessage, AIMessage, SystemMessage # LangChain에서 대화의 각 턴을 나타내는 데 사용되는 객체

history_list = [
    SystemMessage("You are a helpful assistant. Answer all questions to the best of your ability. You must answer in Korean."),  # 챗봇의 역할을 정의
]

In [12]:
while(True):
    user_input = input()

    if user_input == "종료": break
    
    history_list.append(
        HumanMessage(user_input)
    )
    
    ai_msg = chain.invoke(
        {
            "messages": history_list,
        }
    )

    history_list.append(
        AIMessage(ai_msg.content)
    )
    
    print("CHAT_HISTORY:", history_list)
    
    print("AI:", ai_msg.content)

CHAT_HISTORY: [SystemMessage(content='You are a helpful assistant. Answer all questions to the best of your ability. You must answer in Korean.', additional_kwargs={}, response_metadata={}), HumanMessage(content='ㅎㅇ', additional_kwargs={}, response_metadata={}), AIMessage(content='안녕하세요! 만나서 반갑습니다.', additional_kwargs={}, response_metadata={})]
AI: 안녕하세요! 만나서 반갑습니다.
CHAT_HISTORY: [SystemMessage(content='You are a helpful assistant. Answer all questions to the best of your ability. You must answer in Korean.', additional_kwargs={}, response_metadata={}), HumanMessage(content='ㅎㅇ', additional_kwargs={}, response_metadata={}), AIMessage(content='안녕하세요! 만나서 반갑습니다.', additional_kwargs={}, response_metadata={}), HumanMessage(content='뭐함?', additional_kwargs={}, response_metadata={}), AIMessage(content='저는 구글에서 훈련한 대규모 언어 모델입니다.\n\n지금은 사용자님의 질문에 답하고 정보를 제공하며 대화를 나누기 위해 여기에 있습니다. 궁금한 점이 있으시면 언제든지 물어보세요!', additional_kwargs={}, response_metadata={})]
AI: 저는 구글에서 훈련한 대규모 언어 모델입니다.

지금은 사용자님의 질문에

### 2.Chat history

In [ ]:
from langchain_community.chat_message_histories import ChatMessageHistory

chat_history = ChatMessageHistory()

chat_history.add_user_message(
    "내 이름은 김일남이야. 나이는 99세야"
)

chat_history.add_ai_message("그렇군요. 나이가 많으시네요!")

chat_history.messages

[HumanMessage(content='내 이름은 김일남이야. 나이는 99세야', additional_kwargs={}, response_metadata={}),
 AIMessage(content='그렇군요. 나이가 많으시네요!', additional_kwargs={}, response_metadata={})]

In [14]:
chat_history

InMemoryChatMessageHistory(messages=[HumanMessage(content='내 이름은 김일남이야. 나이는 99세야', additional_kwargs={}, response_metadata={}), AIMessage(content='그렇군요. 나이가 많으시네요!', additional_kwargs={}, response_metadata={})])

In [15]:
while(True):
    user_input = input()

    if user_input == "종료": break
    
    chat_history.add_user_message(user_input)
    
    ai_msg = chain.invoke(
        {
            "messages": chat_history.messages,
        }
    )

    chat_history.add_ai_message(ai_msg.content)
    
    print("CHAT_HISTORY:", history_list)
    
    print("AI:", ai_msg.content)

CHAT_HISTORY: [SystemMessage(content='You are a helpful assistant. Answer all questions to the best of your ability. You must answer in Korean.', additional_kwargs={}, response_metadata={}), HumanMessage(content='ㅎㅇ', additional_kwargs={}, response_metadata={}), AIMessage(content='안녕하세요! 만나서 반갑습니다.', additional_kwargs={}, response_metadata={}), HumanMessage(content='뭐함?', additional_kwargs={}, response_metadata={}), AIMessage(content='저는 구글에서 훈련한 대규모 언어 모델입니다.\n\n지금은 사용자님의 질문에 답하고 정보를 제공하며 대화를 나누기 위해 여기에 있습니다. 궁금한 점이 있으시면 언제든지 물어보세요!', additional_kwargs={}, response_metadata={})]
AI: 안녕하세요!
CHAT_HISTORY: [SystemMessage(content='You are a helpful assistant. Answer all questions to the best of your ability. You must answer in Korean.', additional_kwargs={}, response_metadata={}), HumanMessage(content='ㅎㅇ', additional_kwargs={}, response_metadata={}), AIMessage(content='안녕하세요! 만나서 반갑습니다.', additional_kwargs={}, response_metadata={}), HumanMessage(content='뭐함?', additional_kwargs={}, respo

In [18]:
chat_history = ChatMessageHistory()

while(True):
    user_input = input()

    if user_input == "종료": break
    
    chat_history.add_user_message(user_input)
    
    response = chain.invoke(
        {
            "messages": chat_history.messages,
        }
    )
    
    chat_history.add_ai_message(response)

    print("chat_history.messages:", chat_history.messages)
    print("AI:", response.content)

### 3.Automatic history management

In [16]:
prompt = ChatPromptTemplate.from_messages(
    [
        (
            "system",
            "You are a helpful assistant. Answer all questions to the best of your ability. You must answer in Korean.",
        ),
        ("placeholder", "{chat_history}"),
        ("human", "{input}"),
    ]
)

chain = prompt | llm

In [19]:
chain.invoke({
    "chat_history": [],
    "input" : "대한민국의 수도는?"
    }
)

AIMessage(content='대한민국의 수도는 서울입니다.', additional_kwargs={}, response_metadata={'prompt_feedback': {'block_reason': 0, 'safety_ratings': []}, 'finish_reason': 'STOP', 'model_name': 'gemini-2.5-flash', 'safety_ratings': [], 'grounding_metadata': {}, 'model_provider': 'google_genai'}, id='lc_run--548d17ab-c93d-48a4-b5ef-1202b968f717-0', usage_metadata={'input_tokens': 29, 'output_tokens': 70, 'total_tokens': 99, 'input_token_details': {'cache_read': 0}, 'output_token_details': {'reasoning': 62}})

In [20]:
from langchain_core.runnables.history import RunnableWithMessageHistory

# 세션별 채팅 히스토리 관리
chat_histories = {}

# 세션 ID에 따라 대화 기록을 가져오는 함수
def get_session_history(session_id: str):
    if session_id not in chat_histories:
        chat_histories[session_id] = ChatMessageHistory()
    return chat_histories[session_id]

chain_with_message_history = RunnableWithMessageHistory(
    chain, # 실행할 Runnable 객체
    get_session_history, # 세션 ID에 따라 대화 기록을 가져오는 함수
    input_messages_key="input", # 입력 메시지의 Key
    history_messages_key="chat_history", # 대화 히스토리 메시지의 Key
)

In [21]:
config = {"configurable": {"session_id": "kim1"}}  # 세션 ID 설정

chain_with_message_history.invoke(
    {"input": "내 이름은 김일남이야. 나이는 99세야."},
    config=config, # 세션 ID == kim1
)

AIMessage(content='안녕하세요, 김일남 님! 99세이시군요. 만나서 반갑습니다.', additional_kwargs={}, response_metadata={'prompt_feedback': {'block_reason': 0, 'safety_ratings': []}, 'finish_reason': 'STOP', 'model_name': 'gemini-2.5-flash', 'safety_ratings': [], 'grounding_metadata': {}, 'model_provider': 'google_genai'}, id='lc_run--cfc07478-c0dd-4e46-ac7e-184b75499fef-0', usage_metadata={'input_tokens': 39, 'output_tokens': 39, 'total_tokens': 78, 'input_token_details': {'cache_read': 0}, 'output_token_details': {'reasoning': 19}})

In [22]:
chain_with_message_history.invoke(
    {"input": "내 나이는?"}, {"configurable": {"session_id": "kim1"}}
)

AIMessage(content='김일남 님은 99세이십니다.', additional_kwargs={}, response_metadata={'prompt_feedback': {'block_reason': 0, 'safety_ratings': []}, 'finish_reason': 'STOP', 'model_name': 'gemini-2.5-flash', 'safety_ratings': [], 'grounding_metadata': {}, 'model_provider': 'google_genai'}, id='lc_run--9838bffd-fe09-4589-898f-91156e408d65-0', usage_metadata={'input_tokens': 65, 'output_tokens': 45, 'total_tokens': 110, 'input_token_details': {'cache_read': 0}, 'output_token_details': {'reasoning': 31}})

In [23]:
chain_with_message_history.invoke(
    {"input": "내 나이는?"}, {"configurable": {"session_id": "kim2"}}
)

AIMessage(content='죄송합니다만, 저는 인공지능이기 때문에 귀하의 개인 정보를 알 수 없습니다. 따라서 귀하의 나이를 알려드릴 수 없습니다.', additional_kwargs={}, response_metadata={'prompt_feedback': {'block_reason': 0, 'safety_ratings': []}, 'finish_reason': 'STOP', 'model_name': 'gemini-2.5-flash', 'safety_ratings': [], 'grounding_metadata': {}, 'model_provider': 'google_genai'}, id='lc_run--4c8cf034-03b5-4aea-9dd6-84a56504aa26-0', usage_metadata={'input_tokens': 27, 'output_tokens': 106, 'total_tokens': 133, 'input_token_details': {'cache_read': 0}, 'output_token_details': {'reasoning': 74}})

In [24]:
for r in chain_with_message_history.stream(
    {"input": "내가 어느 나라 사람인지 맞춰보고, 그 나라의 문화에 대해 말해봐"},
    config=config,
):
    print(r.content, end="", flush=True)

김일남 님이라는 성함으로 미루어 짐작하건대, **대한민국** 분이실 것 같습니다.

대한민국의 문화에 대해 말씀드리겠습니다.

대한민국은 오랜 역사와 전통을 바탕으로 현대적인 모습이 조화롭게 어우러진 다채로운 문화를 가지고 있습니다.

1.  **유교적 가치**: 한국 문화는 유교의 영향을 많이 받아 어른에 대한 공경(효), 가족 중심주의, 공동체 의식 등을 중요하게 생각합니다. 이러한 가치들은 일상생활과 사회 전반에 깊이 뿌리내려 있습니다.
2.  **음식 문화**: 한국의 음식은 세계적으로도 인정받고 있습니다. 매 끼니 빠지지 않는 김치, 불고기, 비빔밥 등은 대표적인 한식이며, 찌개, 국, 다양한 반찬들이 함께 상에 오르는 풍성한 밥상이 특징입니다. 맵고 얼큰한 맛을 즐기며, 정(情)을 나누는 식사 문화를 중요하게 여깁니다.
3.  **한류(K-Culture)**: 최근 몇 년간 K-팝, K-드라마, K-영화, K-뷰티 등 한국의 대중문화는 전 세계적으로 큰 인기를 얻고 있습니다. 이는 '한류'라고 불리며, 한국의 역동적인 매력을 알리는 데 큰 역할을 하고 있습니다.
4.  **언어와 문자**: 한국어는 독자적인 언어이며, 한글은 세종대왕이 백성들을 위해 창제한 과학적이고 아름다운 문자입니다. 배우기 쉽고 효율적인 문자로 평가받고 있습니다.
5.  **전통과 현대의 조화**: 한복(전통 의상), 한옥(전통 가옥), 판소리 등 전통문화는 여전히 소중히 보존되고 있으며, 동시에 한국은 IT 기술과 첨단 산업을 선도하는 매우 현대적이고 빠르게 변화하는 사회입니다.

이처럼 대한민국은 전통의 깊이와 현대의 활력이 공존하는 매력적인 문화를 가지고 있습니다.

### 4. Modifying chat history

In [25]:
chat_history = ChatMessageHistory()

chat_history.add_user_message("내 이름은 김일남이야.")
chat_history.add_ai_message("안녕하세요, 김일남님! 무엇을 도와드릴까요?")
chat_history.add_user_message("날씨 좋은 날 들을만 한 노래 추천해주세요.")
chat_history.add_ai_message("볼빨간사춘기 - 여행을 추천해요.")

chat_history.messages

[HumanMessage(content='내 이름은 김일남이야.', additional_kwargs={}, response_metadata={}),
 AIMessage(content='안녕하세요, 김일남님! 무엇을 도와드릴까요?', additional_kwargs={}, response_metadata={}),
 HumanMessage(content='날씨 좋은 날 들을만 한 노래 추천해주세요.', additional_kwargs={}, response_metadata={}),
 AIMessage(content='볼빨간사춘기 - 여행을 추천해요.', additional_kwargs={}, response_metadata={})]

In [27]:
prompt = ChatPromptTemplate.from_messages(
    [
        (
            "system",
            "You are a helpful assistant. Answer all questions to the best of your ability. You must answer in Korean.",
        ),
        ("placeholder", "{chat_history}"),
        ("user", "{input}"),
    ]
)

chain = prompt | llm

chain_with_message_history = RunnableWithMessageHistory(
    chain,
    lambda session_id: chat_history, # 단일 사용자 환경
    input_messages_key="input",
    history_messages_key="chat_history",
)

In [28]:

chain_with_message_history.invoke(
    {"input": "내 이름은 뭐야?"},
    {"configurable": {"session_id": "unused"}},
)

AIMessage(content='김일남님입니다.', additional_kwargs={}, response_metadata={'prompt_feedback': {'block_reason': 0, 'safety_ratings': []}, 'finish_reason': 'STOP', 'model_name': 'gemini-2.5-flash', 'safety_ratings': [], 'grounding_metadata': {}, 'model_provider': 'google_genai'}, id='lc_run--a5235f42-928e-4f83-8fbb-03ef09512b5f-0', usage_metadata={'input_tokens': 90, 'output_tokens': 38, 'total_tokens': 128, 'input_token_details': {'cache_read': 0}, 'output_token_details': {'reasoning': 32}})

In [29]:
from langchain_core.runnables import RunnablePassthrough

def summarize_messages(chain_input):
    stored_messages = chat_history.messages

    if len(stored_messages) == 0:
        return False
    
    summarization_prompt = ChatPromptTemplate.from_messages(
        [
            ("placeholder", "{chat_history}"),
            (
                "user",
                "Distill the above chat messages into a single summary message. Include as many specific details as you can. Please, use Korean",
            ),
        ]
    )
    
    summarization_chain = summarization_prompt | llm

    # chat_history 에 저장된 대화 기록을 요약프롬프트에 입력 & 결과 저장
    summary_message = summarization_chain.invoke({"chat_history": stored_messages})

    print("summary_message: ", summary_message)
    
    # chat_history 에 저장되어있던 기록 지우기
    chat_history.clear()

    # 생성된 새로운 요약내용으로 기록 채우기
    chat_history.add_message(summary_message)

    return True


chain_with_summarization = (
    # RunnablePassthrough는 LCEL에서 사용, 입력값을 다음 단계로 그대로 통과시키는 역할
    # assign() 메서드는 체인에 들어오는 딕셔너리에 새로운 키-값 추가
    RunnablePassthrough.assign(messages_summarized=summarize_messages) # 새로운 키 messages_summarized에 값(True 또는 False) 할당, 이 값은 조건부 요약에 활용 가능
    | chain_with_message_history
)

In [30]:
chain_with_summarization.invoke(
    {"input": "내 이름은?"},
    {"configurable": {"session_id": "unused"}},
)

summary_message:  content="사용자분께서는 본인의 이름을 '김일남'이라고 알려주셨습니다. 날씨 좋은 날 들을 만한 노래를 추천해달라고 요청하셨고, 이에 '볼빨간사춘기 - 여행'을 추천해 드렸습니다. 이후 두 차례에 걸쳐 본인의 이름이 무엇인지 다시 질문하셨으며, 챗봇은 일관되게 '김일남님'이라고 답변했습니다." additional_kwargs={} response_metadata={'prompt_feedback': {'block_reason': 0, 'safety_ratings': []}, 'finish_reason': 'STOP', 'model_name': 'gemini-2.5-flash', 'safety_ratings': [], 'grounding_metadata': {}, 'model_provider': 'google_genai'} id='lc_run--781e4176-a802-4c01-a759-b8568433ecc8-0' usage_metadata={'input_tokens': 101, 'output_tokens': 980, 'total_tokens': 1081, 'input_token_details': {'cache_read': 0}, 'output_token_details': {'reasoning': 887}}


AIMessage(content='김일남님이십니다.', additional_kwargs={}, response_metadata={'prompt_feedback': {'block_reason': 0, 'safety_ratings': []}, 'finish_reason': 'STOP', 'model_name': 'gemini-2.5-flash', 'safety_ratings': [], 'grounding_metadata': {}, 'model_provider': 'google_genai'}, id='lc_run--d3966a40-d01b-40e3-a1e6-8600d5ea5cb7-0', usage_metadata={'input_tokens': 121, 'output_tokens': 66, 'total_tokens': 187, 'input_token_details': {'cache_read': 0}, 'output_token_details': {'reasoning': 59}})

In [31]:
chain_with_summarization.invoke(
    {"input": "그 가수는 남자인가요 여자인가요?"},
    {"configurable": {"session_id": "unused"}},
)

summary_message:  content="사용자분께서는 본인의 이름을 '김일남'이라고 알려주셨습니다. 날씨 좋은 날 들을 만한 노래를 추천해달라고 요청하셨고, 이에 '볼빨간사춘기 - 여행'을 추천해 드렸습니다. 이후 두 차례에 걸쳐 본인의 이름이 무엇인지 다시 질문하셨으며, 챗봇은 일관되게 '김일남님'이라고 답변했습니다." additional_kwargs={} response_metadata={'prompt_feedback': {'block_reason': 0, 'safety_ratings': []}, 'finish_reason': 'STOP', 'model_name': 'gemini-2.5-flash', 'safety_ratings': [], 'grounding_metadata': {}, 'model_provider': 'google_genai'} id='lc_run--2a268c27-f539-4462-a283-050818f5f737-0' usage_metadata={'input_tokens': 133, 'output_tokens': 955, 'total_tokens': 1088, 'input_token_details': {'cache_read': 0}, 'output_token_details': {'reasoning': 862}}


AIMessage(content='볼빨간사춘기는 여성 가수입니다. 원래 안지영님과 우지윤님 두 분으로 구성된 여성 듀오였으나, 현재는 안지영님 1인 체제로 활동하고 있습니다.', additional_kwargs={}, response_metadata={'prompt_feedback': {'block_reason': 0, 'safety_ratings': []}, 'finish_reason': 'STOP', 'model_name': 'gemini-2.5-flash', 'safety_ratings': [], 'grounding_metadata': {}, 'model_provider': 'google_genai'}, id='lc_run--37f7a6ec-8b4a-478d-8e9a-2751c58c10e9-0', usage_metadata={'input_tokens': 129, 'output_tokens': 236, 'total_tokens': 365, 'input_token_details': {'cache_read': 0}, 'output_token_details': {'reasoning': 189}})